# Medical Document Analysis with Strands Agent

This notebook demonstrates how to use Strands Agent to:
1. Extract text from PDF medical documents
2. Infer key medical information (diagnosis, medications, treatments)
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)

## Setup and Dependencies


### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude 3.7 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket

Let's now install the requirement packages for our Strands Agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

In [ ]:
pip install --upgrade strands-agents


In [ ]:
pip install strands-agents-tools strands-agents-builder

In [ ]:

!pip show strands-agents-tools

In [ ]:
!pip show strands-agents

# Define Medical Agent
The Medical Document Processing Assistant is an AI-powered tool designed to extract, analyze, and enrich medical information from various document formats such as PDFs and images. This assistant specializes in processing clinical notes, pathology reports, discharge summaries, and other medical documents to provide structured data with standardized medical coding. We will be using Strands to define the agent. 

![arch](architecture.png)

We will defining tools to determine and infer ICD10 code, RxNorm and SNOMED. We are calling the following API within the tool to make the determination. 

1. ICD-10-CM & ICD-10-PCS (U.S. Versions)
Official Source: U.S. Centers for Medicare & Medicaid Services (CMS) and the National Center for Health Statistics (NCHS)

ICD-10-CM (diagnoses):

Call API: https://clinicaltables.nlm.nih.gov/apidoc/icd10cm/v3/doc.html


ICD-10-PCS (procedures):

https://www.cms.gov/medicare/icd-10/2025-icd-10-pcs
(Adjust year as needed)

2. RxNorm
Official Source: U.S. National Library of Medicine (NLM)

https://lhncbc.nlm.nih.gov/RxNav/APIs/RxNormAPIs.html?_gl=1*1qdlo6u*_ga*ODQ1ODkzMzMyLjE3NDg4MzYwMjc.*_ga_7147EPK006*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw*_ga_P1FPTH9PL4*czE3NDg4MzYwMjYkbzEkZzEkdDE3NDg4MzY2MDAkajYwJGwwJGgw

4. SNOMED CT
International Edition

Official Source: SNOMED 

https://browser.ihtsdotools.org/?perspective=full&conceptId1=404684003&edition=MAIN/SNOMEDCT-US/2025-03-01&release=&languages=en



In [ ]:
import os
import json
from typing import Dict, List, Optional, Any
import pypdf
import boto3
from strands import Agent, tool
from strands.models import BedrockModel

In [ ]:
import os
import logging
from strands import Agent
from strands_tools import file_read
from document_processor import process_document
from medical_coding_tools import (
    get_icd, get_rx, get_snomed,
    link_icd, link_rx, link_snomed
)
# System prompt for the medical document processing agent
SYSTEM_PROMPT = """
You are a Medical Document Processing Assistant specialized in extracting and analyzing medical information from clinical documents.

Your tasks include:
1. Processing medical documents (PDFs, images) to extract text
2. Identifying key medical information: diagnoses, medications, treatments
3. Enriching the extracted information with standardized medical codes:
   - ICD-10 codes for diagnoses
   - RxNorm codes for medications
   - SNOMED CT codes for treatments

Provide clear, accurate, and structured information that can be used by healthcare professionals.
"""

# Create the medical document processing agent
medical_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    tools=[
        file_read,
        process_document,
        get_icd,
        get_rx,
        get_snomed,
        link_icd,
        link_rx,
        link_snomed
    ]
)


# Testing

In [ ]:
# Extracted data

# Example clinical note
CLINICAL_NOTE = """
Carlie had a seizure 2 weeks ago. She is complaining of frequent headaches
Nausea is also present. She also complains of eye trouble with blurry vision
Meds : Topamax 50 mgs at breakfast daily,
Send referral order to neurologist
Follow-up as scheduled
"""
# Process the clinical note
response = medical_agent(
        f"Process this clinical note and extract diagnoses, medications, and treatments with their respective medical codes: {CLINICAL_NOTE}"
    )   
print("\nProcessing complete!\n")
print("Usage metrics:")
print(medical_agent.event_loop_metrics)


In [ ]:
# PDF medical document as an input
file_path='p1.pdf'
response = medical_agent(f"Process this medical document and extract diagnoses, medications, and treatments with their respective medical codes: {file_path}")

print("\nProcessing complete!\n")
print("Usage metrics:")
print(medical_agent.event_loop_metrics)

## Conclusion

This notebook demonstrates how to use Strands Agent to:
1. Extract text from PDF medical documents
2. Identify key medical information (diagnoses, medications, treatments)
3. Enrich the data with standardized medical codes (ICD-10, RxNorm, SNOMED CT)

The agent uses a combination of tools to perform these tasks:
- PDF text extraction
- Medical code lookup (ICD-10, RxNorm, SNOMED CT)
- Medical information enrichment

This approach can be extended to handle more complex medical documents and integrate with real medical code databases or APIs.